In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.products"
)

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "unit_price_cast",
        F.col("unit_price").cast("decimal(12,2)")
    )
    .withColumn(
        "cost_price_cast",
        F.col("cost_price").cast("decimal(12,2)")
    )
    .withColumn(
        "launch_date_cast",
        F.to_date("launch_date", "yyyy-MM-dd")
    )
    .withColumn(
        "discontinued_date_cast",
        F.to_date("discontinued_date", "yyyy-MM-dd")
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "product_name_clean",
        F.trim("product_name")
    )
    .withColumn(
        "category_code_clean",
        F.upper(F.trim("category_code"))
    )
    .withColumn(
        "subcategory_code_clean",
        F.upper(F.trim("subcategory_code"))
    )
    .withColumn(
        "brand_name_clean",
        F.trim("brand_name")
    )
    .withColumn(
        "product_status_clean",
        F.upper(F.trim("product_status"))
    )
    .withColumn(
        "currency_code_clean",
        F.upper(F.trim("currency_code"))
    )
)

Define data-quality rules

In [0]:
validated_df = (
    typed_df
    .withColumn(
        "dq_missing_product_id",
        F.col("product_id").isNull()
        | (F.trim(F.col("product_id")) == "")
    )
    .withColumn(
        "dq_missing_product_name",
        F.col("product_name_clean").isNull()
        | (F.col("product_name_clean") == "")
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("product_status_clean").isin(
            "ACTIVE",
            "INACTIVE",
            "DISCONTINUED"
        )
    )
    .withColumn(
        "dq_invalid_unit_price",
        F.col("unit_price_cast").isNull()
        | (F.col("unit_price_cast") < 0)
    )
    .withColumn(
        "dq_invalid_cost_price",
        F.col("cost_price_cast").isNull()
        | (F.col("cost_price_cast") < 0)
    )
    .withColumn(
        "dq_invalid_date_range",
        F.col("launch_date_cast").isNull()
        | (
            F.col("discontinued_date_cast").isNotNull()
            & (
                F.col("discontinued_date_cast")
                < F.col("launch_date_cast")
            )
        )
    )
    .withColumn(
        "dq_invalid_currency",
        ~F.col("currency_code_clean").isin(
            "INR",
            "USD"
        )
    )
)

Add business warnings

In [0]:
validated_df = validated_df.withColumn(
    "dq_warning_selling_below_cost",
    (
        F.col("unit_price_cast").isNotNull()
        & F.col("cost_price_cast").isNotNull()
        & (F.col("unit_price_cast") < F.col("cost_price_cast"))
    )
)

validated_df = validated_df.withColumn(
    "dq_warning_discontinued_without_date",
    (
        (F.col("product_status_clean") == "DISCONTINUED")
        & F.col("discontinued_date_cast").isNull()
    )
)

Define critical failures

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_product_id")
    | F.col("dq_missing_product_name")
    | F.col("dq_invalid_status")
    | F.col("dq_invalid_unit_price")
    | F.col("dq_invalid_cost_price")
    | F.col("dq_invalid_date_range")
    | F.col("dq_invalid_currency")
)

Split valid and rejected rows

In [0]:
reject_df = validated_df.filter(
    F.col("dq_critical_failure")
)

valid_df = validated_df.filter(
    ~F.col("dq_critical_failure")
)

Deduplicate Products

In [0]:
product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(product_window)
    )
    .filter(F.col("_row_number") == 1)
)

Build final Silver table

In [0]:
silver_df = deduped_df.select(
    "product_id",

    F.col("product_name_clean").alias("product_name"),
    F.col("category_code_clean").alias("category_code"),
    F.col("subcategory_code_clean").alias("subcategory_code"),
    F.col("brand_name_clean").alias("brand_name"),

    F.col("unit_price_cast").alias("unit_price"),
    F.col("cost_price_cast").alias("cost_price"),

    F.col("product_status_clean").alias("product_status"),

    F.col("launch_date_cast").alias("launch_date"),
    F.col("discontinued_date_cast").alias("discontinued_date"),

    "supplier_id",

    F.col("currency_code_clean").alias("currency_code"),

    "dq_warning_selling_below_cost",
    "dq_warning_discontinued_without_date",

    F.col("created_at_cast").alias("created_at"),
    F.col("updated_at_cast").alias("updated_at"),

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias(
        "_silver_processed_at"
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.products"
    )
)

Build Quarantine table

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("dq_missing_product_id"),
                F.lit("MISSING_PRODUCT_ID")
            ),
            F.when(
                F.col("dq_missing_product_name"),
                F.lit("MISSING_PRODUCT_NAME")
            ),
            F.when(
                F.col("dq_invalid_status"),
                F.lit("INVALID_PRODUCT_STATUS")
            ),
            F.when(
                F.col("dq_invalid_unit_price"),
                F.lit("INVALID_UNIT_PRICE")
            ),
            F.when(
                F.col("dq_invalid_cost_price"),
                F.lit("INVALID_COST_PRICE")
            ),
            F.when(
                F.col("dq_invalid_date_range"),
                F.lit("INVALID_DATE_RANGE")
            ),
            F.when(
                F.col("dq_invalid_currency"),
                F.lit("INVALID_CURRENCY")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:

if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.product_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.product_rejects ADD COLUMNS (_pipeline_run_id STRING)")

(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.product_rejects"
    )
)

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.products"
).count()

silver_count = spark.table(
    "customer360_dev.silver.products"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.product_rejects"
).count()

print(f"Bronze : {bronze_count:,}")
print(f"Silver : {silver_count:,}")
print(f"Reject : {reject_count:,}")

Bronze : 150,024
Silver : 149,400
Reject : 624
